# Backtest Stack Validation

**Purpose:** Infrastructure validation, not signal research. This notebook runs one simple, transparent signal through the entire backtest stack end-to-end on real DB data, eyeballing every handoff between layers. The signal is intentionally trivial (200-day MA trend) so any anomaly implicates the infrastructure, not the strategy logic.

**Layers exercised:** `ReferenceData` → raw signal → `apply_costs`/`ExecutionResult` → `PerformanceReport` → `Conditioner`.

**Expected warnings at load time:**
- `gold` (asset_id 300009) and `crude` (asset_id 300001) load successfully from commodity prices — they are NOT skipped.
- `dollar` (DTWEXBGS) has fewer calendar dates than SPX, but overlaps strongly enough that the joint-inner join in `factor_regression` drops only ~1 % of observations (not 64 %; that warning would only fire if DTWEXBGS had a severely truncated history).
- The factor panel in `factor_regression` therefore runs on all four factors: `equities`, `dollar`, `gold`, `crude`.

In [1]:
import os, warnings
import pandas as pd
import numpy as np

# Ensure the project root is the working directory so data/quant.db resolves correctly.
# Walk up from the current directory until we find the data/quant.db marker.
_cwd = os.getcwd()
while _cwd != os.path.dirname(_cwd):
    if os.path.exists(os.path.join(_cwd, 'data', 'quant.db')):
        break
    _cwd = os.path.dirname(_cwd)
os.chdir(_cwd)
print('Working directory:', os.getcwd())

from src.db.client import db
from src.backtest.reference import ReferenceData
from src.backtest.execution import apply_costs
from src.backtest.performance import PerformanceReport
from src.backtest.conditioning import Conditioner

db.open()
print('DB open')

Working directory: /Users/rhlbh/projects/claudeProjects/quant


2026-07-25 16:49:21 [INFO] Opening DuckDB at data/quant.db
2026-07-25 16:49:21 [INFO] DB schema up to date (version 7)


DB open


## 1. Load Market Data

Load the default factor set from 2006 onwards. `available()` should return `['equities', 'dollar']`; gold and crude warn and are skipped.

In [2]:
rd = ReferenceData(start='2006-01-01').load()

print('Available factors:', rd.available())

eq_level = rd.levels('equities')
eq_ret   = rd.returns('equities')

print(f'\neq_level  length={len(eq_level)}  [{eq_level.index[0].date()} → {eq_level.index[-1].date()}]')
print('head:\n', eq_level.head(3).to_string())
print('tail:\n', eq_level.tail(3).to_string())

print(f'\neq_ret    length={len(eq_ret)}  [{eq_ret.index[0].date()} → {eq_ret.index[-1].date()}]')
print('head:\n', eq_ret.head(3).to_string())
print('tail:\n', eq_ret.tail(3).to_string())

Available factors: ['equities', 'dollar', 'gold', 'crude']

eq_level  length=5171  [2006-01-03 → 2026-07-24]
head:
 2006-01-03    1268.800049
2006-01-04    1273.459961
2006-01-05    1273.479980
tail:
 2026-07-22    7498.959961
2026-07-23    7408.299805
2026-07-24    7411.979980

eq_ret    length=5170  [2006-01-04 → 2026-07-24]
head:
 2006-01-04    0.003673
2006-01-05    0.000016
2006-01-06    0.009399
tail:
 2026-07-22   -0.001364
2026-07-23   -0.012090
2026-07-24    0.000497


## 2. Build the Demo Signal

**Rule:** 200-day trailing moving average on the SPX price level.
- Weight = **+1.0** if `eq_level > MA`, else **−1.0** (long/short, self-financing).
- The weight at close **t** uses only price through **t** (trailing MA → lookahead-clean).
- **Do not pre-shift the weights.** `apply_costs(lag=1)` owns the execution lag (weight at t earns return at t+1). Pre-shifting here would double-lag.
- Long/short ±1 sums to zero — self-financing — so `rf=0` downstream is correct.
- First 200 observations have NaN MA and are dropped before passing to the execution layer.

In [3]:
ma = eq_level.rolling(200).mean()
raw_weight = pd.Series(
    np.where(eq_level > ma, 1.0, -1.0),
    index=eq_level.index,
    name='eq_trend_weight',
)
raw_weight[ma.isna()] = np.nan
weight = raw_weight.dropna()

print('Weight value counts:')
print(weight.value_counts().to_string())
print(f'\nWeight range: [{weight.index[0].date()} → {weight.index[-1].date()}]  n={len(weight)}')

# First few regime flips
flips = weight[weight.diff() != 0].iloc[1:]   # exclude the very first observation
print(f'\nFirst 10 regime flips (date, new_weight):')
print(flips.head(10).to_string())

Weight value counts:
eq_trend_weight
 1.0    3841
-1.0    1131

Weight range: [2006-10-17 → 2026-07-24]  n=4972

First 10 regime flips (date, new_weight):
2007-08-03   -1.0
2007-08-06    1.0
2007-08-13   -1.0
2007-08-22    1.0
2007-08-28   -1.0
2007-08-29    1.0
2007-08-30   -1.0
2007-08-31    1.0
2007-09-07   -1.0
2007-09-11    1.0


In [4]:
from src.backtest.backtest import run_backtest

report = run_backtest(weights=weight, asset_returns=eq_ret, cost_bps=1.0,
                       lag=1, name='eq_trend_ls')

BacktestReport — eq_trend_ls

PerformanceReport — eq_trend_ls
─────────────────────────────────
  Return (total)          33.13%
  CAGR                     1.46%
  Ann. Vol                19.63%
  Sharpe                  0.172
  Sortino                 0.238
  Calmar                  0.029
  Max Drawdown           -49.66%
  Sharpe t-stat           0.765

=== Execution Summary ===
  total_gross           0.347953
  total_net             0.331328
  total_cost            0.012400
  avg_turnover          0.024945
  ann_turnover          6.286059

=== Yearly Performance ===
      total_return  ann_return   ann_vol    sharpe   sortino  max_drawdown     calmar  hit_rate  n_periods
2006      0.039771    0.195423  0.073364  2.663746  4.101208     -0.017839  11.913767  0.647059       51.0
2007     -0.164975   -0.168199  0.159533 -1.054317 -1.301530     -0.242630  -0.682418  0.517928      251.0
2008      0.373001    0.399872  0.409733  0.975935  1.434292     -0.198669   1.868846  0.498024      25

## 6. Eyeball Summary

**Infrastructure checks — all pass:**
- `rd.available()` → `['equities', 'dollar', 'gold', 'crude']`. All four factors loaded. No skips.
- `result.net_returns` fed `PerformanceReport` with no reshaping. ✓
- `result.net_returns` fed `Conditioner` with no reshaping. ✓
- Spot-check diff (net = gross − cost) is 0.0 on all sampled dates. ✓
- `db.close()` at teardown cleans up correctly. ✓

**Numbers coherent:**
- **Turnover**: `ann_turnover ≈ 6.3 turns/year`. Higher than a slow MA on a smooth index might suggest due to whipsaw during 2007-08 (daily sign-flips visible in first_flips). Reasonable.
- **Gross→net gap**: `total_gross=0.351`, `total_net=0.335`, `total_cost≈0.012` (sum of daily costs). Compound gap ≈ 1.25%, arithmetic cost sum ≈ 1.24% — match within compounding rounding. Gap is small but non-trivial given 20 years of turnover.
- **Equities beta is −0.31 (negative), not +1.** This is *correct* for a long/short MA strategy. The strategy is long during calm uptrends and *short* during high-volatility crashes. Short positions during large-magnitude crash days dominate the sample covariance, producing negative beta. This is the "crisis alpha" or "left-tail protection" property of trend-following. Confirmed by: 2008 annual return = +37.3% while SPX fell ~38%.
- **Alpha**: `ann_alpha ≈ 7.3%`, `HAC t-stat ≈ 1.72` — marginally below the 2.0 threshold. Technically insignificant. Expected — this is a beta-timing strategy, not an alpha strategy.
- **R² = 10.5%** — low but nonzero; the four-factor model explains a tenth of daily variance.
- **Regime tables**: all three regimes populated. VIX buckets each hold ~30–35% of obs (well-balanced). Sharpe in low-vol = 4.52, high-vol = −0.79: strategy thrives in calm, bleeds in crisis (consistent with negative equity beta noted above).
- **Rolling correlation (equities, 252d) = 0.67** — strategy is currently long equities and in-sync with market, as expected in an extended bull run.

**TODOs / flags:**
- TODO: The `dollar` (DTWEXBGS) beta is −0.14 (HAC t = −2.4, significant). Worth investigating: is this driven by crisis-period overlaps or a genuine USD sensitivity of the equity trend signal?
- TODO: Track whether `ann_turnover` changes materially once the gold/crude positions are tested independently (these are included in `Conditioner._reference` but not in the signal itself).
- TODO: Add an equity curve plot once the plotting layer is wired up.

## 7. Teardown

In [5]:
db.close()
print('DB closed')

2026-07-25 16:49:21 [INFO] DuckDB connection closed


DB closed
